# 02 — Fine-Tuned DistilBERT (5-Fold Cross-Validation)

Train `distilbert-base-uncased` on each CV fold and evaluate on the held-out test split.

**Outputs**
- Checkpoints under `results/bert/checkpoints/fold_*`
- Metrics CSVs under `results/bert/`

**Tip:** Set `FOLDS_TO_RUN = [0]` first to verify everything works (~15–30 min), then run all 5 folds.

In [4]:
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bert_model import train_and_evaluate_fold
from src.config import BERT_EPOCHS, BERT_MODEL_NAME, LABELS, N_FOLDS, RESULTS_DIR
from src.data_utils import get_fold_dataframes, prepare_dataset
from src.metrics import summarize_across_folds
from src.results_io import save_all_fold_metrics, save_fold_metrics, save_summary_json

# Start with [0] to test (~10-20 min on GPU), then switch to list(range(N_FOLDS))
FOLDS_TO_RUN = [0]
# FOLDS_TO_RUN = list(range(N_FOLDS))

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    raise RuntimeError(
        "CUDA not available. Select kernel 'Python 3.12 (fuzzy-final)' "
        "and install PyTorch cu128 (required for RTX 5060)."
    )
try:
    torch.randn(1, device="cuda")
except RuntimeError as exc:
    raise RuntimeError(
        "GPU detected but CUDA kernels failed. Install PyTorch cu128:\n"
        "  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128"
    ) from exc

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device} ({torch.cuda.get_device_name(0)})")
print(f"Model: {BERT_MODEL_NAME}")
print(f"Epochs per fold: {BERT_EPOCHS}")
print(f"Folds to run: {FOLDS_TO_RUN}")

Project root: C:\Users\yusuf\OneDrive\Desktop\vscode\FuzzyLogic\final
Device: cuda (NVIDIA GeForce RTX 5060)
Model: distilbert-base-uncased
Epochs per fold: 3
Folds to run: [0]


## 1. Load subsampled data and CV folds

In [5]:
df, folds = prepare_dataset()
fold_dfs = get_fold_dataframes(df, folds)

print(f"Dataset size: {len(df):,}")
for fold_idx, fold in enumerate(fold_dfs):
    print(
        f"Fold {fold_idx}: train={len(fold['train'])}, "
        f"val={len(fold['val'])}, test={len(fold['test'])}"
    )

Dataset size: 25,000
Fold 0: train=18000, val=2000, test=5000
Fold 1: train=18000, val=2000, test=5000
Fold 2: train=18000, val=2000, test=5000
Fold 3: train=18000, val=2000, test=5000
Fold 4: train=18000, val=2000, test=5000


## 2. Train and evaluate each fold

In [ ]:
checkpoint_root = RESULTS_DIR / "bert" / "checkpoints"
checkpoint_root.mkdir(parents=True, exist_ok=True)

bert_results = []

for fold_idx in FOLDS_TO_RUN:
    fold = fold_dfs[fold_idx]
    output_dir = str(checkpoint_root / f"fold_{fold_idx}")

    print(f"\n=== Fold {fold_idx} ===")
    result = train_and_evaluate_fold(
        fold_idx=fold_idx,
        train_df=fold["train"],
        val_df=fold["val"],
        test_df=fold["test"],
        output_dir=output_dir,
        device=device,
    )

    save_fold_metrics(fold_idx, "bert", result.metrics, output_dir=RESULTS_DIR / "bert")

    bert_results.append(
        {
            "fold": fold_idx,
            "metrics": result.metrics,
        }
    )

    print(f"Fold {fold_idx} macro F1: {result.metrics['macro_f1']:.4f}")
    print(f"Fold {fold_idx} micro F1: {result.metrics['micro_f1']:.4f}")


=== Fold 0 ===


KeyboardInterrupt: 

## 3. Results summary

In [ ]:
save_all_fold_metrics(bert_results, method="bert", output_dir=RESULTS_DIR / "bert")
save_summary_json(bert_results, method="bert", output_dir=RESULTS_DIR / "bert")

summary_rows = []
for item in bert_results:
    summary_rows.append(
        {
            "fold": item["fold"],
            "macro_f1": item["metrics"]["macro_f1"],
            "micro_f1": item["metrics"]["micro_f1"],
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print("\nPer-label metrics (mean ± std across folds):")
display(summarize_across_folds([r["metrics"] for r in bert_results]))

print(f"\nSaved to: {RESULTS_DIR / 'bert'}")